# Phase 3+4 integration — 5-seed sweep, success_threshold=0.3 (Colab)

Runs `experiments/19_phase34_integrated.py` for seeds {17, 11, 23, 1, 2} on CUDA.

**Variant from default:** `--success-threshold 0.3` (default is 0.5). Report 028 found the Hebbian updater fired on only ~1% of cues at threshold 0.5 — most cues had q either ≥ 0.95 (trivial-skip) or < 0.5 (failure). Lowering to 0.3 catches the partial-success band so the codebook actually drifts. Result paths use the `hebbian_st03` tag so they don't collide with the report-028 run.

**Prereqs:**
1. Codebook at `MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt` (already there from report 028 run).
2. Runtime → Change runtime type → A100 (or T4).
3. Run cells top-to-bottom (or Runtime → Run all).

Results sync to `MyDrive/neuro-ai/results/phase34_integrated_hebbian_st03_*/`. Cell 7 prints a summary table including Hebbian consolidations and codebook drift — those are the critical diagnostic numbers this run is meant to move.

In [ ]:
# 1. Clone the repo (public or private — for private, configure a token first).
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI

In [ ]:
# 2. Mount Drive and copy the codebook into place.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps. Colab ships torch+CUDA; we just need torchhd and datasets.
!pip install -q torchhd datasets

In [ ]:
# 4. Sanity check: GPU present, FHRR + codebook load.
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

import sys; sys.path.insert(0, 'src')
from energy_memory.phase2.persistence import load_codebook
cb = load_codebook('reports/phase3c_reconstruction/phase3c_codebook_reconstruction.pt', device='cuda')
print('codebook:', cb.shape, cb.dtype, cb.device)

In [ ]:
# 5. Run 5-seed sweep sequentially. Logs and per-seed result dirs live under reports/.
# Variant: --success-threshold 0.3 (default is 0.5) to widen the Hebbian firing band.
# Report 028 found firing rate ~1% at threshold 0.5; this run is meant to drive it up
# so the codebook actually drifts and Phase 4 has a moving target to discover against.
import subprocess, os, time
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
RUN_TAG = 'hebbian_st03'
log_root = Path(f'reports/phase34_integrated_{RUN_TAG}_5seed_colab')
log_root.mkdir(parents=True, exist_ok=True)

for seed in [17, 11, 23, 1, 2]:
    out_dir = f'reports/phase34_integrated_{RUN_TAG}_seed{seed}'
    log_path = log_root / f'seed{seed}.log'
    print(f'\n=== seed {seed} -> {out_dir} ===  ({time.strftime("%H:%M:%S")})')
    with open(log_path, 'w') as logf:
        proc = subprocess.run(
            ['python', 'experiments/19_phase34_integrated.py',
             '--updater-kind', 'hebbian',
             '--success-threshold', '0.3',
             '--seed', str(seed),
             '--device', 'cuda',
             '--output-dir', out_dir],
            stdout=logf, stderr=subprocess.STDOUT,
        )
    print(f'    exit={proc.returncode}  log={log_path}')
    !tail -8 {log_path}

In [ ]:
# 6. Copy results back to Drive so the VM dying doesn't lose them.
import shutil, os
RUN_TAG = 'hebbian_st03'
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for seed in [17, 11, 23, 1, 2]:
    src = f'reports/phase34_integrated_{RUN_TAG}_seed{seed}'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/phase34_integrated_{RUN_TAG}_seed{seed}', dirs_exist_ok=True)
shutil.copytree(f'reports/phase34_integrated_{RUN_TAG}_5seed_colab',
                f'{dst}/phase34_integrated_{RUN_TAG}_5seed_colab', dirs_exist_ok=True)
print('results in', dst)
!ls {dst}

In [ ]:
# 7. Quick summary table across seeds — final-checkpoint metrics + Hebbian firing diagnostics.
import json
from pathlib import Path

RUN_TAG = 'hebbian_st03'
print(f'{"seed":>4} {"cond":>16} {"top1":>8} {"R@10":>8} {"cap_t05":>8} {"cands":>6} {"cons":>5} {"drift":>9}')
for seed in [17, 11, 23, 1, 2]:
    p = Path(f'reports/phase34_integrated_{RUN_TAG}_seed{seed}/phase34_results.json')
    if not p.exists(): continue
    d = json.loads(p.read_text())
    for cond in ('baseline_static', 'phase3_reencode', 'phase3_phase4'):
        final = d['results'][cond][-1]
        cands = final.get('candidates_total', 0)
        cons = final.get('consolidations', 0)
        drift = final.get('codebook_drift_from_initial', 0.0)
        print(f'{seed:>4} {cond:>16} {final["top1"]:>8.3f} {final["topk"]:>8.3f} '
              f'{final["cap_t_05"]:>8.3f} {cands:>6d} {cons:>5d} {drift:>9.2e}')